## Target Variable Creation

The original dataset did not include a risk level (Low, Medium or High). To create this target the total number of disease cases was calculated by adding the reported diarrheal, cholera and typhoid cases for each record. 
The total disease cases were then divided into three groups using the 33rd and 67th percentiles of the data:

Low Risk - Lowest one-third of the total disease cases
Medium Risk - Middle one-third of the total disease cases
High Risk - Highest one-third of the total disease cases

This method was chosen because the dataset did not provide official risk level thresholds. Using percentiles creates balanced groups which helps improve the performance of machine learning model.

In [1]:
import pandas as pd

In [2]:
#Load the cleaned dataset
df = pd.read_csv(
    "../data/processed/clean_water_pollution_disease.csv",
    keep_default_na=False
)

In [3]:
df.head()

,country,region,year,water_source_type,contaminant_level,ph,turbidity,dissolved_oxygen,nitrate,lead,...,cholera_cases,typhoid_cases,infant_mortality_rate,gdp_per_capita,healthcare_access,urbanization,sanitation_coverage,rainfall,temperature,population_density
0,Mexico,North,2015,Lake,1.04,6.88,0.15,9.38,10.91,3.03,...,1,4,13.77,11855,59.52,85.96,97.36,676,16.48,80
1,Brazil,West,2017,Well,2.48,7.26,1.06,6.73,21.71,4.67,...,2,5,19.08,10434,80.14,84.78,94.37,1796,23.11,25
2,Indonesia,Central,2022,Pond,3.38,7.58,7.03,4.67,22.55,8.32,...,9,35,35.79,4559,54.05,51.23,76.44,1672,24.76,114
3,Nigeria,East,2016,Well,2.50,6.86,2.40,7.88,16.29,5.01,...,2,8,74.65,2293,46.38,58.27,48.95,1015,25.97,241
4,Mexico,South,2005,Well,0.73,6.52,2.24,7.80,8.91,2.00,...,1,2,18.73,10765,71.50,91.83,87.58,453,25.09,52


In [4]:
df.shape

(3000, 24)

In [5]:
#Double check the cleaned column names
df.columns.tolist()

['country',
 'region',
 'year',
 'water_source_type',
 'contaminant_level',
 'ph',
 'turbidity',
 'dissolved_oxygen',
 'nitrate',
 'lead',
 'bacteria_count',
 'water_treatment_method',
 'clean_water_access',
 'diarrheal_cases',
 'cholera_cases',
 'typhoid_cases',
 'infant_mortality_rate',
 'gdp_per_capita',
 'healthcare_access',
 'urbanization',
 'sanitation_coverage',
 'rainfall',
 'temperature',
 'population_density']

In [6]:
#Create total disease cases
df["total_disease_cases"] = (
    df["diarrheal_cases"]
    + df["cholera_cases"]
    + df["typhoid_cases"]
)

In [7]:
df[
    [
        "diarrheal_cases",
        "cholera_cases",
        "typhoid_cases",
        "total_disease_cases"
    ]
].head(10)

,diarrheal_cases,cholera_cases,typhoid_cases,total_disease_cases
0,25,1,4,30
1,63,2,5,70
2,197,9,35,241
3,88,2,8,98
4,39,1,2,42
5,66,2,5,73
6,59,0,10,69
7,82,2,2,86
8,78,2,11,91
9,132,0,9,141


In [8]:
#Check the total disease distribution
df["total_disease_cases"].describe()

count    3000.000000
mean      105.145333
std        77.912676
min         4.000000
25%        45.000000
50%        87.000000
75%       148.000000
max       539.000000
Name: total_disease_cases, dtype: float64

In [9]:
lower_threshold = df["total_disease_cases"].quantile(1 / 3)
upper_threshold = df["total_disease_cases"].quantile(2 / 3)

print("Low/Medium threshold:", lower_threshold)
print("Medium/High threshold:", upper_threshold)

Low/Medium threshold: 59.0
Medium/High threshold: 124.0


In [10]:
#Create Low, Medium and High risk labels
def assign_risk_level(total_cases):
    if total_cases <= lower_threshold:
        return "Low"
    elif total_cases <= upper_threshold:
        return "Medium"
    else:
        return "High"

In [11]:
df["risk_level"] = df["total_disease_cases"].apply(assign_risk_level)

In [12]:
#Check the new target
df[
    [
        "total_disease_cases",
        "risk_level"
    ]
].head(15)

,total_disease_cases,risk_level
0,30,Low
1,70,Medium
2,241,High
3,98,Medium
4,42,Low
5,73,Medium
6,69,Medium
7,86,Medium
8,91,Medium
9,141,High


In [13]:
risk_counts = df["risk_level"].value_counts()

print(risk_counts)

risk_level
Low       1009
High       999
Medium     992
Name: count, dtype: int64


In [14]:
risk_percentages = (
    df["risk_level"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(risk_percentages)

risk_level
Low       33.63
High      33.30
Medium    33.07
Name: proportion, dtype: float64


In [15]:
df.groupby("risk_level")["total_disease_cases"].agg(
    ["count", "min", "max", "mean", "median"]
).sort_values("mean")

,count,min,max,mean,median
risk_level,,,,,
Low,1009,4,59,32.295342,31.0
Medium,992,60,124,88.809476,87.0
High,999,125,539,194.945946,177.0


In [16]:
print("Missing risk labels:", df["risk_level"].isna().sum())

Missing risk labels: 0


In [17]:
#Save the final labelled dataset
df.to_csv(
    "../data/final/waterborne_disease_risk_dataset.csv",
    index=False
)

In [18]:
import os

print(
    os.path.exists(
        "../data/final/waterborne_disease_risk_dataset.csv"
    )
)

True


In [19]:
print(lower_threshold)
print(upper_threshold)
print(df["risk_level"].value_counts())

59.0
124.0
risk_level
Low       1009
High       999
Medium     992
Name: count, dtype: int64
